In [1]:
import pandas as pd

df = pd.read_csv("../data/raw/US_Accidents_March23.csv", nrows=5000)

print("Boyut:", df.shape)
print()
print("İlçe örnekleri:", df["County"].head(10).tolist())
print()
print("Şiddet dağılımı:")
print(df["Severity"].value_counts())

Boyut: (5000, 46)

İlçe örnekleri: ['Montgomery', 'Franklin', 'Clermont', 'Montgomery', 'Montgomery', 'Franklin', 'Montgomery', 'Montgomery', 'Montgomery', 'Franklin']

Şiddet dağılımı:
Severity
2    2938
3    2053
4       5
1       4
Name: count, dtype: int64


In [2]:
# Sütun isimleri ve tipleri
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 5000 entries, 0 to 4999
Data columns (total 46 columns):
 #   Column                 Non-Null Count  Dtype  
---  ------                 --------------  -----  
 0   ID                     5000 non-null   str    
 1   Source                 5000 non-null   str    
 2   Severity               5000 non-null   int64  
 3   Start_Time             5000 non-null   str    
 4   End_Time               5000 non-null   str    
 5   Start_Lat              5000 non-null   float64
 6   Start_Lng              5000 non-null   float64
 7   End_Lat                0 non-null      float64
 8   End_Lng                0 non-null      float64
 9   Distance(mi)           5000 non-null   float64
 10  Description            5000 non-null   str    
 11  Street                 5000 non-null   str    
 12  City                   5000 non-null   str    
 13  County                 5000 non-null   str    
 14  State                  5000 non-null   str    
 15  Zipcode        

In [3]:
pd.set_option("display.max_rows", 100)

eksik = pd.DataFrame({
    "tip": df.dtypes,
    "dolu": df.notna().sum(),
    "eksik_yuzde": (df.isna().mean() * 100).round(1)
})
print(eksik.to_string())

                           tip  dolu  eksik_yuzde
ID                         str  5000          0.0
Source                     str  5000          0.0
Severity                 int64  5000          0.0
Start_Time                 str  5000          0.0
End_Time                   str  5000          0.0
Start_Lat              float64  5000          0.0
Start_Lng              float64  5000          0.0
End_Lat                float64     0        100.0
End_Lng                float64     0        100.0
Distance(mi)           float64  5000          0.0
Description                str  5000          0.0
Street                     str  5000          0.0
City                       str  5000          0.0
County                     str  5000          0.0
State                      str  5000          0.0
Zipcode                    str  5000          0.0
Country                    str  5000          0.0
Timezone                   str  5000          0.0
Airport_Code               str  5000          0.0


In [4]:
# Sadece eksiği olan sütunlar, çoktan aza sıralı
eksik_olanlar = eksik[eksik.eksik_yuzde > 0].sort_values("eksik_yuzde", ascending=False)
print(eksik_olanlar.to_string())

                       tip  dolu  eksik_yuzde
End_Lat            float64     0        100.0
End_Lng            float64     0        100.0
Precipitation(in)  float64   192         96.2
Wind_Chill(F)      float64   460         90.8
Wind_Speed(mph)    float64  4541          9.2
Visibility(mi)     float64  4958          0.8
Weather_Condition      str  4964          0.7
Weather_Timestamp      str  4981          0.4
Temperature(F)     float64  4978          0.4
Humidity(%)        float64  4978          0.4
Pressure(in)       float64  4980          0.4
Wind_Direction         str  4981          0.4


In [1]:
import pandas as pd

YOL = "../data/raw/US_Accidents_March23.csv"

parcalar = []
okuyucu = pd.read_csv(YOL, usecols=["State", "Severity"], chunksize=1_000_000)

for i, parca in enumerate(okuyucu, 1):
    print(f"{i}. parça okundu")
    parcalar.append(parca.groupby("State")["Severity"].value_counts().unstack(fill_value=0))

tablo = pd.concat(parcalar).groupby(level=0).sum()
tablo["TOPLAM"] = tablo.sum(axis=1)
print(tablo.sort_values("TOPLAM", ascending=False).head(20).to_string())

1. parça okundu
2. parça okundu
3. parça okundu
4. parça okundu
5. parça okundu
6. parça okundu
7. parça okundu
8. parça okundu
Severity        1        2         3      4     TOPLAM
State                                                 
CA        10284.0  1445833  271814.0  13502  1741433.0
FL         7083.0   755895  104065.0  13149   880192.0
TX         4233.0   450952  120443.0   7209   582837.0
SC         6175.0   330817   42470.0   3095   382557.0
NY         1247.0   265902   70062.0  10749   347960.0
NC         5139.0   293066   29590.0  10404   338199.0
VA         3115.0   230660   51324.0  18202   303301.0
PA         1731.0   247991   31031.0  15867   296620.0
MN          489.0   160848   29884.0    863   192084.0
OR         1323.0   162713    9730.0   5894   179660.0
AZ         7389.0   140553   17551.0   5116   170609.0
GA         1007.0    94206   61245.0  12776   169234.0
IL         1620.0   105273   57748.0   4317   168958.0
TN         2335.0   133054   28510.0   3489   1

In [2]:
import pandas as pd

YOL = "../data/raw/US_Accidents_March23.csv"
EYALETLER = ["FL", "NY", "MN"]

ATILACAK = ["End_Lat", "End_Lng", "Wind_Chill(F)", "Distance(mi)",
            "End_Time", "Description", "Country", "Source",
            "Turning_Loop", "Weather_Timestamp", "Airport_Code"]

parcalar = []
okuyucu = pd.read_csv(YOL, chunksize=500_000, low_memory=False)

for i, parca in enumerate(okuyucu, 1):
    secili = parca[parca["State"].isin(EYALETLER)]
    parcalar.append(secili)
    print(f"{i}. parça — {len(secili):,} satır alındı")

df = pd.concat(parcalar, ignore_index=True)
df = df.drop(columns=ATILACAK)

print()
print("Toplam satır:", f"{len(df):,}")
print("Sütun sayısı:", df.shape[1])
print()
print(df["State"].value_counts())

1. parça — 74,548 satır alındı
2. parça — 70,126 satır alındı
3. parça — 75,798 satır alındı
4. parça — 76,230 satır alındı
5. parça — 75,437 satır alındı
6. parça — 68,603 satır alındı
7. parça — 76,381 satır alındı
8. parça — 94,775 satır alındı
9. parça — 115,995 satır alındı
10. parça — 115,243 satır alındı
11. parça — 116,719 satır alındı
12. parça — 122,991 satır alındı
13. parça — 123,131 satır alındı
14. parça — 121,793 satır alındı
15. parça — 59,776 satır alındı
16. parça — 32,690 satır alındı

Toplam satır: 1,420,236
Sütun sayısı: 35

State
FL    880192
NY    347960
MN    192084
Name: count, dtype: int64


In [3]:
# --- 1. Tarihi datetime'a çevir ---
df["Start_Time"] = pd.to_datetime(df["Start_Time"], format="mixed")

# --- 2. Yağış: eksik = yağmur yoktu -> 0 ---
df["Precipitation(in)"] = df["Precipitation(in)"].fillna(0)

# --- 3. Sayısal sütunları medyanla doldur ---
sayisal = ["Temperature(F)", "Humidity(%)", "Pressure(in)",
           "Visibility(mi)", "Wind_Speed(mph)"]
for s in sayisal:
    df[s] = df[s].fillna(df[s].median())

# --- 4. Metin sütunlarını "Bilinmiyor" ile doldur ---
metin = ["Weather_Condition", "Wind_Direction", "City", "Zipcode", "Street"]
for m in metin:
    df[m] = df[m].fillna("Bilinmiyor")

# --- 5. İlçe referans tablosunu bağla ---
ref = pd.read_csv("../data/raw/county_referans.csv")
df["county_key"] = df["County"].str.lower().str.replace(".", "", regex=False).str.strip()
df = df.merge(ref[["State", "county_key", "nufus_2022",
                   "nufus_yogunlugu", "kentsel_kirsal"]],
              on=["State", "county_key"], how="left")

# --- Kontrol ---
print("Satır:", f"{len(df):,}")
print("Kalan eksik değerler:")
print(df.isna().sum()[df.isna().sum() > 0].to_string())
print()
print("Eşleşmeyen ilçe oranı:", f"{df['nufus_2022'].isna().mean():.2%}")

Satır: 1,420,236
Kalan eksik değerler:
Timezone                  801
Sunrise_Sunset           5395
Civil_Twilight           5395
Nautical_Twilight        5395
Astronomical_Twilight    5395
nufus_2022                139
nufus_yogunlugu           139
kentsel_kirsal            139

Eşleşmeyen ilçe oranı: 0.01%


In [4]:
# Kalan metin eksiklerini doldur
for s in ["Sunrise_Sunset", "Civil_Twilight",
          "Nautical_Twilight", "Astronomical_Twilight", "Timezone"]:
    df[s] = df[s].fillna("Bilinmiyor")

# İlçesi eşleşmeyen 139 satırı sil
oncesi = len(df)
df = df.dropna(subset=["nufus_2022"])
print(f"Silinen satır: {oncesi - len(df)}")

print("Kalan eksik değer:", df.isna().sum().sum())
print("Son boyut:", df.shape)

Silinen satır: 139
Kalan eksik değer: 0
Son boyut: (1420097, 39)


In [5]:
import re

# --- ZAMAN DEĞİŞKENLERİ ---
df["saat"]       = df["Start_Time"].dt.hour
df["gun"]        = df["Start_Time"].dt.dayofweek        # 0=Pazartesi
df["ay"]         = df["Start_Time"].dt.month
df["yil"]        = df["Start_Time"].dt.year
df["hafta_sonu"] = df["gun"] >= 5
df["yogun_saat"] = df["saat"].isin([7, 8, 9, 16, 17, 18])

# Mevsim
def mevsim(ay):
    if ay in [12, 1, 2]:  return "Kis"
    if ay in [3, 4, 5]:   return "Ilkbahar"
    if ay in [6, 7, 8]:   return "Yaz"
    return "Sonbahar"

df["mevsim"] = df["ay"].apply(mevsim)

# --- YOL TİPİ ---
def yol_tipi(s):
    s = str(s)
    if re.search(r"\bI-\d+", s):                          return "Otoyol"
    if re.search(r"\bUS-\d+|US Highway", s):              return "Federal yol"
    if re.search(r"State (Route|Rte)|\b[A-Z]{2}-\d+", s): return "Eyalet yolu"
    if re.search(r"County (Hwy|Road)|\bCR-\d+", s):       return "Ilce yolu"
    if re.search(r"\b(Dr|Ave|St|Ln|Ct|Blvd|Rd|Way|Pl)\b", s): return "Sehir ici"
    return "Diger"

df["yol_tipi"] = df["Street"].apply(yol_tipi)

# --- KONTROL ---
print("Yol tipi dağılımı:")
print(df["yol_tipi"].value_counts())
print()
print("Mevsim dağılımı:")
print(df["mevsim"].value_counts())
print()
print("Yeni boyut:", df.shape)

Yol tipi dağılımı:
yol_tipi
Sehir ici      638123
Diger          386553
Otoyol         298637
Federal yol     45875
Eyalet yolu     44900
Ilce yolu        6009
Name: count, dtype: int64

Mevsim dağılımı:
mevsim
Kis         435350
Sonbahar    383512
Ilkbahar    308431
Yaz         292804
Name: count, dtype: int64

Yeni boyut: (1420097, 47)


In [6]:
print(df[df["yol_tipi"] == "Diger"]["Street"].value_counts().head(30).to_string())

Street
Florida's Tpke S          9647
Brooklyn Queens Expy      9549
Florida's Tpke N          9065
Palmetto Expy S           7508
Long Island Expy W        7501
Long Island Expy E        7261
 S Orange Blossom Trl     6798
 S Dixie Hwy              6351
Palmetto Expy N           5911
Florida's Tpke            4448
Southern State Pkwy E     3937
Major Deegan Expy N       3386
Southern State Pkwy W     3365
Ronald Reagan Tpke        3360
W Beltway S               3232
Dolphin Expy E            3153
 S Tamiami Trl            3062
Adirondack Northway N     2845
E Beltway N               2829
New York State Thruway    2755
Adirondack Northway S     2635
 S John Young Pkwy        2531
W Beltway N               2530
Grand Central Pkwy E      2480
New England Trwy S        2461
E Beltway S               2383
Taconic State Pkwy        2327
Long Island Expy          2285
Belt Pkwy E               2274
Van Wyck Expy N           2220


In [7]:
def yol_tipi(s):
    s = str(s)
    # Numaralı eyaletler arası otoyol
    if re.search(r"\bI-\d+", s):
        return "Otoyol"
    # İsimli hızlı yollar
    if re.search(r"\b(Expy|Expressway|Tpke|Turnpike|Pkwy|Parkway|Fwy|Freeway|Thruway|Beltway|Northway|Skyway|Causeway)\b", s, re.I):
        return "Otoyol"
    if re.search(r"\bUS-\d+|US Highway", s):
        return "Federal yol"
    if re.search(r"State (Route|Rte)|\b[A-Z]{2}-\d+", s):
        return "Eyalet yolu"
    if re.search(r"County (Hwy|Road)|\bCR-\d+", s):
        return "Ilce yolu"
    if re.search(r"\b(Hwy|Highway)\b", s, re.I):
        return "Federal yol"
    if re.search(r"\b(Dr|Drive|Ave|Avenue|St|Street|Ln|Lane|Ct|Court|Blvd|Boulevard|Rd|Road|Way|Pl|Place|Pike|Trl|Trail|Cir|Circle|Ter|Terrace|Loop|Row|Aly|Alley|Sq|Square|Plz|Plaza|Bridge|Tunnel)\b", s, re.I):
        return "Sehir ici"
    return "Diger"

df["yol_tipi"] = df["Street"].apply(yol_tipi)

print(df["yol_tipi"].value_counts())
print()
print("Diger oranı:", f"{(df['yol_tipi'] == 'Diger').mean():.1%}")
print()
print("Hâlâ sınıflanmayanlar:")
print(df[df["yol_tipi"] == "Diger"]["Street"].value_counts().head(15).to_string())

yol_tipi
Sehir ici      690271
Otoyol         518920
Federal yol    116739
Eyalet yolu     44900
Diger           43258
Ilce yolu        6009
Name: count, dtype: int64

Diger oranı: 3.0%

Hâlâ sınıflanmayanlar:
Street
New England Trwy S             2461
Bilinmiyor                     1899
New York Trwy W                1772
George Washington Brg          1733
Central Florida Greeneway S    1330
New York Trwy N                1296
 Route 9                        834
Shakopee Byp N                  790
New England Trwy N              771
Central Florida Greeneway N     768
Route 9                         702
Central Florida GreeneWay       665
Brooklyn Brg                    651
New England Thwy                638
S Broadway                      509


In [8]:
print(df.iloc[0].to_string())

ID                                              A-116062
Severity                                               3
Start_Time                           2016-11-30 15:36:03
Start_Lat                                      27.981367
Start_Lng                                     -82.326561
Street                   E Dr Martin Luther King Jr Blvd
City                                               Tampa
County                                      Hillsborough
State                                                 FL
Zipcode                                            33610
Timezone                                      US/Eastern
Temperature(F)                                      80.6
Humidity(%)                                         70.0
Pressure(in)                                       29.94
Visibility(mi)                                      10.0
Wind_Direction                                       SSW
Wind_Speed(mph)                                      5.8
Precipitation(in)              